# FeatureValidator Demo

This notebook demonstrates the new `FeatureValidator` class for walkforward testing and permutation testing.

## Key Features

1. **Portfolio-Agnostic Design**: Always accepts a Portfolio instance
2. **Walkforward Testing**: Validates portfolio performance across time periods
3. **Permutation Testing**: Tests significance using two-region shuffling
4. **Simple API**: User creates portfolio, validator tests it

## Workflow

1. Load data (features, targets, candles)
2. Create a portfolio (single-feature or multi-ensemble)
3. Run walkforward test
4. Analyze results
5. Run permutation test
6. Visualize significance

In [ ]:
import sys
from pathlib import Path

# Setup project path
project_root = Path.cwd() if (Path.cwd() / 'vault').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

# Core imports
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Feature validator
from feature_selection.feature_validator import FeatureValidator

# Feature extraction
from feature_extraction.feature_extractor import extract_features_with_forward_returns
from utils.core.helpers import load_data_multi_ticker

# Portfolio creation
from ensemble.portfolio import Portfolio
from ensemble.diversified_ensemble import DiversifiedEnsemble
from ensemble.vault_manager import load_ensemble_from_vault

# Utilities
from utils.core.enums import TimeFrame, Ticker

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

print("✓ Imports successful")
print(f"Project root: {project_root}")

## 1. Extract Features and Targets

We'll use the `feature_extractor` module to:
1. Extract features (e.g., RSI signals)
2. Automatically compute forward returns (targets)
3. Handle ATR/EWSD normalization for volatility scaling
4. Ensure proper alignment between features and targets

The extractor handles all the complexity of:
- Forward return calculation (Feature[t] predicts Return[t+1])
- ATR/EWSD extraction for volatility scaling
- Proper index alignment

In [ ]:
# Extract features and targets using feature extractor
# This automatically handles:
# - Feature extraction (RSI signals)
# - ATR/EWSD extraction (for volatility scaling)
# - Forward returns computation (shifted by 1 period)
# - Proper alignment

# Single ticker example (uncomment to use):
# tickers = Ticker.ES

# Multi-ticker example:
tickers = [Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY ]  # Multiple tickers

start_date = datetime(2000, 1, 1)
end_date = datetime(2024, 12, 31)

print("Extracting features and targets...")
print(f"Tickers: {[t.name if hasattr(t, 'name') else str(t) for t in (tickers if isinstance(tickers, list) else [tickers])]}")

features_df, targets_df = extract_features_with_forward_returns(
    module_name='rsi',
    params={'lookback': 2},  # RSI with 14-period lookback
    ticker=tickers,  # Can be single ticker or list
    start=start_date,
    end=end_date,
    timeframes=[TimeFrame.D],
    target_col='log_return'  # Use ATR-normalized returns (recommended for multi-ticker)
)

print(f"\n✓ Features extracted: {len(features_df)} rows")
print(f"  Feature columns: {[col for col in features_df.columns if col != 'ticker']}")
print(f"\n✓ Targets computed: {len(targets_df)} rows")
print(f"  Target columns: {[col for col in targets_df.columns if col != 'ticker']}")

# Show ticker distribution
if 'ticker' in features_df.columns:
    print(f"\n  Tickers: {features_df['ticker'].unique()}")
    print(f"  Samples per ticker:")
    print(features_df['ticker'].value_counts().sort_index())

print(f"\n  Date range: {features_df.index.min().date()} to {features_df.index.max().date()}")

# Display feature distribution (for first RSI column found)
rsi_cols = [col for col in features_df.columns if 'rsi' in col.lower() and col != 'ticker']
if rsi_cols:
    print(f"\nFeature distribution ({rsi_cols[0]}):")
    print(features_df[rsi_cols[0]].value_counts())

print(f"\nReturn statistics:")
print(targets_df['log_return_ewsd'].describe())

features_df.head(10)

In [ ]:
# Load candles for portfolio fitting/prediction
# The feature extractor handles features/targets, but we still need candles
# for Portfolio.fit_from_candles() and Portfolio.predict_from_candles()

# Normalize tickers to list format
ticker_list = tickers if isinstance(tickers, list) else [tickers]

candles_df = load_data_multi_ticker(
    tickers=ticker_list,
    timeframe=TimeFrame.D,
    start=start_date,
    end=end_date
)

print(f"\n✓ Candles loaded: {len(candles_df)} rows")
print(f"  Date range: {candles_df['datetime'].min().date()} to {candles_df['datetime'].max().date()}")
print(f"  Columns: {list(candles_df.columns)}")

# Show ticker distribution in candles
if 'ticker' in candles_df.columns:
    print(f"\n  Tickers: {candles_df['ticker'].unique()}")
    print(f"  Samples per ticker:")
    print(candles_df['ticker'].value_counts().sort_index())

candles_df.head(10)

In [ ]:
# Verify alignment between features, targets, and candles
print("\nData Alignment Check:")
print("="*70)
print(f"Features shape: {features_df.shape}")
print(f"Targets shape: {targets_df.shape}")
print(f"Candles shape: {candles_df.shape}")

# Check that features and targets are aligned
if len(features_df) == len(targets_df):
    print("✓ Features and targets are aligned")
else:
    print("⚠ Warning: Features and targets have different lengths")

# Check index alignment
if features_df.index.equals(targets_df.index):
    print("✓ Features and targets have matching indices")
else:
    print("⚠ Warning: Features and targets have different indices")

targets_df.head(10)

## 2. Create Portfolio

The user is responsible for constructing the portfolio. Here we create a minimal single-feature portfolio with:
- Single ensemble
- Single base model
- RSI signal feature

**Multi-Ticker Support:**
- The portfolio automatically handles multiple tickers
- Each base model processes all tickers in the portfolio
- Portfolio aggregates signals across tickers using instrument weights
- ATR/EWSD normalization ensures fair comparison across tickers with different volatility levels

In [ ]:
import tempfile
import json
import os

# Get the actual feature column name from extracted features
feature_cols = [col for col in features_df.columns if col != 'ticker']
if len(feature_cols) == 0:
    raise ValueError("No feature columns found in features_df")
feature_col = feature_cols[0]  # Use first feature column

# Get ticker names from features_df
if 'ticker' in features_df.columns:
    unique_tickers = features_df['ticker'].unique().tolist()
else:
    # Fallback: use ticker_list from above
    unique_tickers = [t.name if hasattr(t, 'name') else str(t) for t in ticker_list]

print(f"Using feature column: {feature_col}")
print(f"Tickers: {unique_tickers}")

def create_single_feature_portfolio(
    feature_col,
    strategy='long',
    target_volatility=0.20,
    tickers=None
):
    """
    Create a minimal portfolio with single ensemble and single base model.
    
    Supports both single-ticker and multi-ticker portfolios.
    
    This is a helper function - in production, you'd construct portfolios
    using your standard pipeline or load from vault.
    """
    if tickers is None:
        tickers = ['ES']  # Default
    
    # Normalize tickers to list of strings
    if isinstance(tickers, str):
        tickers = [tickers]
    elif isinstance(tickers, list):
        tickers = [t.name if hasattr(t, 'name') else str(t) for t in tickers]
    
    # Create base model config
    base_model_config = {
        'name': feature_col,
        'feature_column': feature_col,
        'model_type': 'QuantileBinningModel',
        'strategy': strategy,
        'constructor_params': {'n_bins': 3},
        'bias_node_spec': {
            'module_name': 'rsi',
            'timeframes': ['D'],
            'params': {'lookback': 14}
        },
        'tickers': tickers
    }
    
    # Create temporary control file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
        temp_control_file = {
            'metadata': {
                'created_at': datetime.now().isoformat(),
                'is_fit': False,
                'base_tf': 'D'
            },
            'base_models': [base_model_config],
            'tickers': tickers
        }
        json.dump(temp_control_file, f, indent=2)
        temp_path = f.name
    
    try:
        # Create ensemble from control file
        ensemble = DiversifiedEnsemble(
            target_volatility=target_volatility,
            control_file_path=temp_path
        )
    finally:
        # Clean up temp file
        os.unlink(temp_path)
    
    # Create portfolio with single ensemble
    portfolio = Portfolio(
        ensembles=[ensemble],
        trading_timeframe=TimeFrame.D,
        target_volatility=target_volatility
    )
    
    return portfolio

# Create portfolio with multiple tickers
portfolio = create_single_feature_portfolio(
    feature_col=feature_col,
    strategy='long',
    target_volatility=0.20,
    tickers=unique_tickers  # Pass all tickers from features
)

print("\n✓ Portfolio created")
print(f"  Ensembles: {len(portfolio.ensembles)}")
print(f"  Trading timeframe: {portfolio.trading_timeframe}")
print(f"  Target volatility: {portfolio.target_volatility}")
print(f"  Base models: {len(portfolio.ensembles[0].base_models)}")
print(f"  Feature column: {feature_col}")
print(f"  Tickers: {unique_tickers}")

## 3. Run Walkforward Test

The validator repeatedly fits and predicts on the portfolio across time-series folds.

In [ ]:
# Initialize validator
validator = FeatureValidator(features_df, targets_df)

print(validator)
print(f"\nData ready for validation:")
print(f"  Features: {validator.n_features}")
print(f"  Samples: {validator.n_samples}")
print(f"  Date range: {validator.date_range[0].date()} to {validator.date_range[1].date()}")

In [ ]:
# Determine target column (use ATR-normalized for multi-ticker, log_return for single)
target_col = 'log_return_ewsd' if len(unique_tickers) > 1 else 'log_return'
if target_col not in targets_df.columns:
    target_col = 'log_return'  # Fallback

print(f"Using target column: {target_col}")

# Run walkforward test
fold_results, summary_df, aggregate_metrics = validator.walkforward_test(
    portfolio=portfolio,
    candles_df=candles_df,
    train_start=datetime(2010, 1, 1),
    train_end=datetime(2015, 1, 1),
    test_step=252,  # ~1 year test periods
    num_steps=5,    # 5 walkforward steps
    target_col=target_col,
    verbose=True
)

print("\n" + "="*70)
print("WALKFORWARD TEST COMPLETE")
print("="*70)

## 4. Analyze Walkforward Results

Let's examine the per-fold performance and aggregate metrics.

In [ ]:
# Display summary DataFrame
print("Per-Fold Performance:")
print("="*70)
display_cols = ['fold', 'test_start', 'test_end', 'sharpe_ratio', 'sortino_ratio', 
                'total_return', 'max_drawdown', 'n_trades']
print(summary_df[display_cols].to_string(index=False))

print("\n\nAggregate Metrics (Across All Folds):")
print("="*70)
for metric, value in aggregate_metrics.items():
    if isinstance(value, (int, float)):
        print(f"{metric:20s}: {value:10.4f}")

# Plot per-fold Sharpe ratios
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(summary_df['fold'], summary_df['sharpe_ratio'], alpha=0.7)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.axhline(y=aggregate_metrics['sharpe_ratio'], color='green', linestyle='--', 
           label=f'Aggregate: {aggregate_metrics["sharpe_ratio"]:.2f}')
ax.set_xlabel('Fold')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Per-Fold Sharpe Ratios')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot cumulative returns across all folds
all_strategy_returns = pd.concat([r['strategy_returns'] for r in fold_results])
all_baseline_returns = pd.concat([r['baseline_returns'] for r in fold_results])

# Calculate cumulative returns
strategy_cumulative = (1 + all_strategy_returns).cumprod()
baseline_cumulative = (1 + all_baseline_returns).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(strategy_cumulative.index, strategy_cumulative.values, 
        label='Strategy', linewidth=2, alpha=0.8)
ax.plot(baseline_cumulative.index, baseline_cumulative.values, 
        label='Baseline (Buy & Hold)', linewidth=2, alpha=0.8)

# Add fold boundaries
for i, fold in enumerate(fold_results):
    if i > 0:  # Skip first line
        ax.axvline(x=fold['test_start'], color='gray', linestyle=':', alpha=0.3)

ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return')
ax.set_title('Cumulative Returns: Strategy vs Baseline')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal cumulative return:")
print(f"  Strategy: {(strategy_cumulative.iloc[-1] - 1) * 100:.2f}%")
print(f"  Baseline: {(baseline_cumulative.iloc[-1] - 1) * 100:.2f}%")

## 5. Run Walkforward Permutation Test

Test if the portfolio's performance is statistically significant by comparing against permuted data.

**Two-Region Shuffling:**
- Region 1: First training window `[train_start, train_end)`
- Region 2: Remaining data `[train_end, data_end)`

Each region is shuffled independently to maintain temporal structure within walkforward folds.

In [ ]:
# Run permutation test (use same target column as walkforward test)
perm_results = validator.walkforward_permutation_test(
    portfolio=portfolio,
    candles_df=candles_df,
    train_start=datetime(2010, 1, 1),
    train_end=datetime(2015, 1, 1),
    test_step=252,
    num_steps=5,
    target_col=target_col,  # Use same target column as walkforward test
    nreps=50,  # 50 permutations (increase for production)
    alpha=0.05,
    n_jobs=-1,  # Use all CPUs
    shuffle_target=False,  # Shuffle features (default)
    verbose=True
)

print("\n" + "="*70)
print("PERMUTATION TEST COMPLETE")
print("="*70)

## 6. Analyze Permutation Test Results

Interpret the statistical significance of the portfolio's performance.

In [ ]:
# Display results
print("Permutation Test Results:")
print("="*70)
print(perm_results.to_string(index=False))

print("\n\nInterpretation:")
print("="*70)
for _, row in perm_results.iterrows():
    feature = row['feature']
    pval = row['pval']
    significant = row['significant']
    original = row['original_criterion']
    
    if significant:
        print(f"✓ {feature}")
        print(f"  - Original Sharpe: {original:.4f}")
        print(f"  - P-value: {pval:.4f} (significant at α=0.05)")
        print(f"  - Conclusion: Feature has genuine predictive power")
    else:
        print(f"✗ {feature}")
        print(f"  - Original Sharpe: {original:.4f}")
        print(f"  - P-value: {pval:.4f} (not significant)")
        print(f"  - Conclusion: Performance may be due to chance")

## Optional: Examine Individual Folds

Dig deeper into specific fold performance.

In [ ]:
# Examine fold 0 in detail
fold_idx = 0
fold = fold_results[fold_idx]

print(f"Fold {fold_idx} Details:")
print("="*70)
print(f"Train: {fold['train_start'].date()} to {fold['train_end'].date()} ({fold['n_train']} samples)")
print(f"Test:  {fold['test_start'].date()} to {fold['test_end'].date()} ({fold['n_test']} samples)")

print(f"\nMetrics:")
for metric, value in fold['metrics'].items():
    if isinstance(value, (int, float)):
        print(f"  {metric:20s}: {value:10.4f}")

# Plot fold returns
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Daily returns
fold['strategy_returns'].plot(ax=ax1, label='Strategy', alpha=0.7)
fold['baseline_returns'].plot(ax=ax1, label='Baseline', alpha=0.7)
ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=0.5)
ax1.set_title(f'Fold {fold_idx}: Daily Returns')
ax1.set_ylabel('Return')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cumulative returns
(1 + fold['strategy_returns']).cumprod().plot(ax=ax2, label='Strategy', linewidth=2)
(1 + fold['baseline_returns']).cumprod().plot(ax=ax2, label='Baseline', linewidth=2)
ax2.set_title(f'Fold {fold_idx}: Cumulative Returns')
ax2.set_xlabel('Date')
ax2.set_ylabel('Cumulative Return')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show positions
print(f"\nPositions (first 10):")
print(fold['positions_df'].head(10))

## Multi-Ticker Notes

**Key Points:**
- ✅ Multi-ticker extraction is already demonstrated above
- Use `log_return_ewsd` or `log_return_ewsd` for volatility scaling (recommended)
- Portfolio automatically handles multiple tickers
- Feature extractor uses millisecond offsets to avoid duplicate indices
- Each ticker's features are extracted independently and then combined

## 7. Summary and Next Steps

The `FeatureValidator` provides a clean, portfolio-agnostic API for walkforward testing and permutation testing.

In [ ]:
### Key Takeaways

**Simplified API:**
- Always accepts a `Portfolio` instance
- User constructs portfolio beforehand
- Validator just repeatedly calls `fit_from_candles()` and `predict_from_candles()`

**Walkforward Testing:**
- Validates portfolio performance across time-series folds
- Provides per-fold and aggregate metrics
- No look-ahead bias

**Permutation Testing:**
- Tests statistical significance using two-region shuffling
- Maintains walkforward integrity
- P-values indicate if performance is genuine vs random

**Next Steps:**

1. **Try Different Features:**
   - Test multiple RSI lookbacks
   - Test momentum, trend-following signals
   - Test ensemble portfolios

2. **Increase Permutation Reps:**
   - Use 100-1000 replications for production
   - More reliable p-values

3. **Multiple Tickers:**
   - Test multi-instrument portfolios
   - Examine diversification benefits

4. **Custom Metrics:**
   - Currently uses Sharpe ratio
   - Could modify criterion function for other metrics

5. **Load from Vault:**
   - Test production portfolios from vault
   - Validate existing strategies

### Example: Testing Vault Portfolio

```python
# Load portfolio from vault
from ensemble.vault_manager import load_ensemble_from_vault

ensemble = load_ensemble_from_vault('vault/D/rsi_long')
portfolio = Portfolio(
    ensembles=[ensemble],
    trading_timeframe=TimeFrame.D,
    target_volatility=0.20
)

# Run walkforward test
results = validator.walkforward_test(
    portfolio=portfolio,
    candles_df=candles_df,
    train_start=datetime(2010, 1, 1),
    train_end=datetime(2015, 1, 1)
)
```

print("Demo complete! ✓")

In [1]:
import sys
from pathlib import Path
# Get project root (current working directory when running from project root)
# If running from research/, use parent; otherwise use cwd()
project_root = Path.cwd() if (Path.cwd() / 'vault').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))